# 1주차 5교시 — 조건 선택·정제·집계·파생변수 만들기

## 학생용 지적 실험 노트

**수업이 끝나면 할 수 있는 것:** 의도적으로 섞인 회원 데이터의 표기·자료형·결측·중복을 정리하고 업무 질문에 답합니다.

> 실행 전에 반드시 예상 답을 적습니다. `___` 또는 `FILL_ME`만 채운 뒤 실행하고, 값을 한 번 바꾸고, 결과가 왜 달라졌는지 한 문장으로 적습니다.


## 지금 열 파일과 수업 진행 방법

- 지금 열 파일: `1주차_5교시_학생용_지적실험노트.ipynb`
- 함께 받은 `matchmaking_data/01_week1` 폴더의 CSV를 사용합니다. 폴더 위치는 바꾸지 않습니다.
- 강사가 PPT에서 Q번호를 보여주면 노트북에서 같은 Q번호를 찾습니다.
- 먼저 결과를 예상한 뒤 빈칸을 채우고 실행합니다.
- 교시가 끝나면 자신이 작성한 노트북을 저장합니다.

강사용 정답 노트북과 `03_전체실행.py`는 수업 중에 열지 않습니다. `03_전체실행.py`는 강사가 전체 코드와 환경을 미리 확인하는 파일입니다.


## 50분 운영 리듬

- 0~5분 · 질문을 행·열·요약으로 분해
- 5~14분 · 문자열 표기 통일
- 14~23분 · 숫자 변환·결측
- 23~31분 · 중복 점검
- 31~40분 · query·boolean
- 40~47분 · groupby·apply·np.where
- 47~50분 · 도움 없이 새 문제 풀기

## 오늘 다룰 범위

- **학생 직접 실습:** query, boolean selection, value_counts, nunique, fillna, dropna, replace, astype/pd.to_numeric, drop, groupby, mean/sum/count, reset_index, apply, np.where
- **강사 시연·학생 해석:** agg, duplicated, drop_duplicates, str.strip/lower/split, map
- **선택 부록·추후 학습:** transform, 복잡한 apply(axis=1), pivot·melt, 정규표현식


## 준비 셀 — 데이터와 라이브러리 연결

- **뜻:** 오늘 사용할 도구와 데이터 폴더를 준비합니다.
- **이유:** 뒤의 Q가 데이터 위치와 앞 교시에 의존하지 않게 만듭니다.
- **정상:** 데이터 폴더 경로가 출력됩니다.
- **복구:** 프로젝트 폴더 안에서 노트북을 열고 이 셀부터 다시 실행합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_week1_data_dir():
    """어디에서 노트북을 열어도 수업 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "01_week1",
            base / "matchmaking_data" / "01_week1",
            base / "01_week1",
        ]
        for candidate in candidates:
            if (candidate / "matchmaking_members.csv").exists():
                return candidate
    raise FileNotFoundError("matchmaking_members.csv를 찾지 못했습니다. 프로젝트 폴더 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week1_data_dir()
print("데이터 폴더:", DATA_DIR)


## 수업에서 사용할 데이터 준비

이 셀은 미리 제공하는 준비 코드입니다. 학생은 따라 치지 않고 실행 결과의 행·열 수만 확인합니다.


In [ ]:
members_raw = pd.read_csv(DATA_DIR / "matchmaking_members.csv")
members = members_raw.copy()
print("원본:", members.shape)


## Q1. 업무 질문을 ‘행·열·요약’ 세 조각으로 번역하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: 긴 자연어 질문을 대상 행, 볼 열, 요약 방법으로 나눕니다.
> 
> 이유: 코드를 한 줄 통째로 외우지 않고 질문을 단계별로 번역하기 위해서입니다.
> 
> 정상: ‘활성이고 30세 이상 / 지역·로그인 / 지역별 수·평균’으로 나뉩니다.
> 
> 복구: 문장에서 ‘~인 회원’, ‘무엇’, ‘~별 평균/개수’를 각각 표시합니다.

### 실행 전에 생각할 질문

질문: 활성이고 30세 이상인 회원의 지역별 수와 4주 평균 로그인은?

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
target_rows = "FILL_ME"
needed_columns = "FILL_ME"
summary_method = "FILL_ME"
print(target_rows, needed_columns, summary_method)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q2. 지역 표기의 앞뒤 공백과 영문 표기를 통일하세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: str.strip은 양끝 공백을 지우고 replace는 같은 뜻의 값을 하나로 통일합니다.
> 
> 이유: ‘서울’, ‘서울 ’, ‘Seoul’, ‘seoul’이 서로 다른 그룹으로 세어지는 문제를 막습니다.
> 
> 정상: 정제 후 서울 244, 경기 231이고 영문·공백 표기가 사라집니다.
> 
> 복구: 원본을 덮어쓰기 전 copy를 만들고 value_counts로 전후를 비교합니다.

### 실행 전에 생각할 질문

실행 전 value_counts에서 이상한 지역명이 몇 종류인지 찾으세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
print(members["residence_region"].value_counts().tail(8))
members["residence_region"] = members["residence_region"].str.___()
members["residence_region"] = members["residence_region"].___({"Seoul": "서울", "seoul": "서울"})
print(members["residence_region"].value_counts().head())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q3. 연봉의 ‘미정’을 숫자 결측으로 바꾸고 중앙값으로 채우세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: pd.to_numeric(errors='coerce')는 숫자로 못 바꾸는 값을 NaN으로 만들고 fillna로 정한 원칙을 적용합니다.
> 
> 이유: object 연봉에는 평균·크기 비교를 안전하게 적용할 수 없기 때문입니다.
> 
> 정상: 변환 후 결측 11개, 중앙값 대체 후 0개이며 dtype은 float입니다.
> 
> 복구: 바로 astype(int)하지 말고 변환 실패값을 먼저 출력해 원인을 확인합니다.

### 실행 전에 생각할 질문

빈칸을 평균과 중앙값 중 무엇으로 채울지 선택하고, 극단값에 덜 흔들리는 중앙값을 사용합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
income_numeric = pd.to_numeric(members["annual_income_10k_krw"], errors=___)
print("변환 후 결측:", income_numeric.isna().sum())
income_median = income_numeric.___()
members["annual_income_10k_krw"] = income_numeric.___(income_median)
print(members["annual_income_10k_krw"].dtype, members["annual_income_10k_krw"].isna().sum())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q4. 중복 회원 3행은 무엇을 기준으로 찾고 제거할까요?

> **처음 배우는 사람을 위한 안내**
>
> 뜻: duplicated는 같은 key가 다시 나온 행을 표시하고 drop_duplicates는 정책에 따라 하나만 남깁니다.
> 
> 이유: 중복이 그룹 수·평균과 다음 교시 merge의 행 수를 부풀리기 때문입니다.
> 
> 정상: member_id 중복 추가행 3개, 제거 후 1000행입니다.
> 
> 복구: 전체 행이 아니라 회원 식별자인 member_id를 subset으로 사용했는지 확인합니다.

### 실행 전에 생각할 질문

keep='first'는 먼저 가입한 행을 남긴다는 수업 규칙입니다. 실제 업무에서는 원천시각·수정시각 정책을 확인해야 합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
duplicate_count = members.___(subset="member_id").sum()
print("추가 중복 행:", duplicate_count)
members = members.___(subset="member_id", keep="first").copy()
print(members.shape)


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q5. query와 불리언 조건으로 같은 대상을 고르세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: query는 문자열 질문, 불리언 선택은 True인 행만 통과시키는 방식입니다.
> 
> 이유: 전체 회원에서 업무 조건을 만족하는 대상만 집계하기 위해서입니다.
> 
> 정상: 두 방식의 member_id가 완전히 같습니다.
> 
> 복구: query 안에서는 ==, 불리언 조건 여러 개는 각각 괄호로 감싸고 &를 씁니다.

### 실행 전에 생각할 질문

‘활성이고 30세 이상’을 두 방식으로 쓰고 결과 수가 같은지 비교하세요.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
selected_query = members.___("is_active == True and age >= 30")
selected_boolean = members[(members["is_active"] == True) ___ (members["age"] >= 30)]
print(len(selected_query), len(selected_boolean))
print((selected_query["member_id"].sort_values().to_numpy() == selected_boolean["member_id"].sort_values().to_numpy()).all())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q6. 지역별 회원 수와 평균 로그인을 집계하고 활동 수준을 만드세요

> **처음 배우는 사람을 위한 안내**
>
> 뜻: groupby는 같은 지역끼리 묶고 agg는 여러 요약을 한 번에 만듭니다. apply는 함수, np.where는 두 갈래 조건을 열 전체에 적용합니다.
> 
> 이유: 행 목록을 의사결정에 쓸 수 있는 지역별 요약과 상담 라벨로 바꾸기 위해서입니다.
> 
> 정상: summary는 지역별 행, member_count·avg_logins 열을 가집니다. apply와 where 결과가 같습니다.
> 
> 복구: 먼저 login_total 열을 만든 뒤 groupby의 묶을 열과 집계할 열을 소리 내어 읽습니다.

### 실행 전에 생각할 질문

단순 두 갈래는 np.where, 복잡한 재사용 규칙은 함수+apply가 읽기 쉽다는 선택 기준을 말합니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
login_cols = ["login_week1", "login_week2", "login_week3", "login_week4"]
members["login_total"] = members[login_cols].___(axis=1)
region_summary = (members.groupby("residence_region")
                  .agg(member_count=("member_id", "count"), avg_logins=("login_total", "mean"))
                  .___())

def activity_label(total):
    return "활발" if total >= 10 else "일반"

members["label_apply"] = members["login_total"].___(activity_label)
members["label_where"] = np.___(members["login_total"] >= 10, "활발", "일반")
print(region_summary.sort_values("member_count", ascending=False).head())
print((members["label_apply"] == members["label_where"]).all())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## Q7. 앞에서 배운 코드로 VIP 활동 회원 보고서 완성하기

> **처음 배우는 사람을 위한 안내**
>
> 성공 조건: 응답률 결측 행은 dropna로 제외하고, VIP·활성·로그인 10회 이상을 고른 뒤 지역별 수와 평균 응답률을 만듭니다.
> 
> 빈칸 안내: 결측 행 제외, 조건 선택, 지역별 묶기와 집계에 필요한 기능을 채웁니다.
> 
> 정상: 결과 열은 residence_region, member_count, avg_response_rate입니다.
> 
> 복구: 각 조건을 하나씩 적용해 행 수가 언제 0이 되는지 확인합니다.

### 실행 전에 생각할 질문

fillna가 아니라 dropna인 이유는 이번 질문에서 응답률 평균의 실제 관측값만 사용한다는 명시적 정책 때문입니다.

### 학생이 할 일

1. 결과를 먼저 적습니다.
2. 핵심 빈칸만 채웁니다.
3. 실행 결과를 예상과 비교합니다.
4. 숫자·조건·열 하나를 바꿔 다시 실행합니다.
5. 결과가 달라진 이유를 한 문장으로 설명합니다.


In [ ]:
# 1. 응답률이 비어 있는 행을 제외하고 사본을 만듭니다.
vip_source = members.___(
    subset=["recommendation_response_rate_prior"]
).copy()

# 2. VIP·활성·로그인 10회 이상인 회원을 선택합니다.
vip_active = vip_source.___(
    "membership_tier == 'VIP' and is_active == True and login_total >= 10"
)

# 3. 지역별 회원 수와 평균 응답률을 만듭니다.
vip_report = (vip_active.___("residence_region")
    .agg(
        member_count=("member_id", "___"),
        avg_response_rate=("recommendation_response_rate_prior", "___")
    )
    .___()
)

print(vip_report.head())


### 내 설명

- 예상과 실제가 같았나: 
- 값을 바꾸자 무엇이 달라졌나: 
- 이 기능을 쓰지 못하면 생길 문제: 
- 막혔을 때 처음 확인한 것: 


## 마무리 질문

1. 오늘 가장 자주 쓸 기능 하나와 그 이유는?
2. 오늘 기능을 쓰게 되는 데이터의 신호는?
3. 정상 결과를 무엇으로 확인했나?
4. 다음 교시로 넘겨야 할 결과는?
